In [1]:
import os

import pandas as pd

import data_util
import duckdb
import plot_util
import pyarrow.parquet as pq
import seaborn as sns
import wandb
import sys
import json
import numpy as np
import sys

In [22]:
data_root = os.path.abspath("./data/wandb")
# group = "ersac_testing"
group = "vlite_testing"
project = "jamesr-j/BSuite_Testing"

# if not os.path.exists(os.path.join(data_root, group)):
if os.path.exists(os.path.join(data_root, group)):
    api = wandb.Api(timeout=2000)
    runs = api.runs(project, filters={"group": group})
    assert runs
    save_dir = os.path.join(data_root, group)
    os.makedirs(save_dir, exist_ok=True)
    data_util.write_runs_to_parquet(runs, save_dir)
    # for run in runs:
    #     table = data_util._run_to_arrow_table(run, save_metadata=True)
    #     save_path = os.path.join(save_dir, f"{run.id}.parquet")
    #     print("Saving", save_path)
    #     pq.write_table(table, save_path)

file_glob = os.path.join(data_root, group)  # , "*.parquet")

def read_wandb_metadata(path: os.PathLike, filename: bool = True):
    parquet_metadata = pq.read_metadata(path)
    wandb_metadata = json.loads(parquet_metadata.metadata[b"wandb"])
    if filename:
        wandb_metadata["filename"] = os.fspath(path)
    return wandb_metadata

all_dicts = []
for file in os.listdir(file_glob):
    meta_step = data_util.read_wandb_metadata(os.path.join(file_glob, file))
    all_dicts.append(meta_step)
meta = pd.DataFrame(all_dicts)

### below is dodgy workaround when differeing lengths config, to keep the struct datatype in duckdb
max_keys = 0
max_keys_id = None
for row_id in range(len(meta["config"])):
    length = len(meta["config"][row_id])
    if length > max_keys:
        max_keys = length
        max_keys_id = row_id
all_keys = meta["config"][max_keys_id].keys()

def update_dict(d):
  return {key: d.get(key, np.nan) for key in all_keys}

meta["config"] = meta["config"].apply(update_dict)
### and here it ends

# meta = data_util.read_wandb_metadata(file_glob)  # TODO this is the original
metrics = data_util.read_wandb_history(os.path.join(file_glob, "*.parquet"))
# measurements = duckdb.from_df(meta.loc[:, ["id", *meta.attrs["sweep_vars"]]]).join(metrics, "id")  # TODO the original

# print(metrics)
# sys.exit()

measurements = duckdb.from_df(meta.loc[:, ["id", "config"]]).join(metrics, "id")
# measurements = duckdb.from_df(meta.loc[:, ["id"]]).join(metrics, "id")

Skip writing runs.
